import stuff

In [47]:
import torchvision
# For image transforms
from torchvision import transforms
# For DATA SET
import torchvision.datasets as datasets
# For Pytorch methods
import torch
import torch.nn as nn
# For Optimizer
import torch.optim as optim
# FOR DATA LOADER
from torch.utils.data import DataLoader
# FOR TENSOR BOARD VISUALIZATION
from torch.utils.tensorboard import SummaryWriter # to print to tensorboard
import copy




hyperparams

In [48]:
# Hyperparameters
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
lr = 3e-4
batchSize = 32  # Batch size
numEpochs = 100
logStep = 625  # the number of steps to log the images and losses to tensorboard

latent_dimension = 128 # 64, 128, 256
# for simplicity we will flatten the image to a vector and to use simple MLP networks
# 28 * 28 * 1 flattens to 784
# you are also free to use CNNs
image_dimension = 28 * 28 * 1  # 784

fixed_noise = torch.randn(batchSize, latent_dimension).to(device)


normalization and dataloading

In [49]:
# we define a tranform that converts the image to tensor and normalizes it with mean and std of 0.5
# which will convert the image range from [0, 1] to [-1, 1]
myTransforms = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.5,), (0.5,))])

# the MNIST dataset is available through torchvision.datasets
print("loading MNIST digits dataset")
dataset = datasets.MNIST(root="dataset/", transform=myTransforms, download=True)
# let's create a dataloader to load the data in batches
loader = DataLoader(dataset, batch_size=batchSize, shuffle=True)


loading MNIST digits dataset


Generator model

In [50]:
class Generator(nn.Module):
    """
    Generator Model
    """
    def __init__(self, latent_dimension):
        super().__init__()
        self.gen = nn.Sequential(
            nn.Linear(latent_dimension, 128),
            nn.ReLU(inplace=True),

            nn.Linear(128, 256),
            nn.ReLU(inplace=True),

            nn.Linear(256, 784),
            nn.Tanh() # It is helpful to use the tanh activation function to force the ouput into the [-1,1] range that our normalized images have.
        )

    def forward(self, x):
        return self.gen(x)

Discriminator

In [51]:
class Discriminator(nn.Module):
    """
    Discriminator Model
    """
    def __init__(self):
        super().__init__()
        self.disc = nn.Sequential(
            nn.Linear(784, 256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(256, 128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.disc(x)

initialize network and optimizers

In [52]:
# initialize networks and optimizers
discriminator = Discriminator().to(device)
generator = Generator(latent_dimension).to(device)
opt_discriminator = optim.Adam(discriminator.parameters(), lr=lr)
opt_generator = optim.Adam(generator.parameters(), lr=lr)

# This is a binary classification task, so we use Binary Cross Entropy Loss
criterion = nn.BCELoss()

writer_fake = SummaryWriter(f"runs/GAN/fake")
writer_real = SummaryWriter(f"runs/GAN/real")


In [53]:
best_generator_loss = float("inf")
patience = 10
patience_counter = 0

best_generator_state = None
best_discriminator_state = None

Training

In [54]:

# Training Loop
step = 0
print("Started Training and visualization...")
for epoch in range(numEpochs):
    epoch_generator_loss = 0.0
    num_batches=0

    # loop over batches
    print()
    for batch_idx, (real, _) in enumerate(loader):
        # First we train the discriminator on real images vs. generated images

        # Get the real images and flatten them
        # for simplicity, we flatten the image to a vector and to use simple MLP networks
        # 28 * 28 * 1 flattens to 784
        real = real.view(-1, 784).to(device)
        batch_size = real.shape[0]

        # Step 1) generate fake images
        noise = torch.randn(batch_size, latent_dimension).to(device)
        fake = generator(noise)

        # Step 2) Train Discriminator:

        # - real images are labeled as 1
        # -fake images are labeled as 0


        real_labels = torch.ones(batch_size, 1, device=device)
        fake_labels = torch.zeros(batch_size, 1, device=device)

        # - predict the discriminator output for real images
        discriminator_real = discriminator(real)
        # - calculate the loss for real images
        loss_real = criterion(discriminator_real, real_labels)

        # - predict the discriminator output for fake images
        discriminator_fake = discriminator(fake)
        # -calculate the loss for fake images
        loss_fake = criterion(discriminator_fake, fake_labels)

        # -average the loss for real and fake images
        loss_discriminator = (loss_real + loss_fake) / 2

        # - now upadate the weights of the discriminator by backpropagating the loss through the discriminator

        # the generator is not updated in this step
        # HINT: call the `backward` method of the discriminator with the argument `retain_graph=True` to keep the computational graph
        # this is necessary because we will use the same discriminator to train the generator
        opt_discriminator.zero_grad()
        loss_discriminator.backward(retain_graph=True)
        opt_discriminator.step()

        # Train Generator:
        # Now train the generator by generating fake images and passing them through the discriminator
        # You can do a little trick and modify the original objective function of
        # "minimizing the probability of the discriminator predicting the fake images as fake"
        # to "maximizing the probability of the discriminator predicting the fake images as real"
        # this leads to a faster training of the generator when it does not represent the real data well
        # this is a common trick in GANs
        # for more information see section 17.1.2 of the book Deep Learning by Bishop and Bishop

        generator_predictions = discriminator(fake)
        loss_generator = criterion(generator_predictions, real_labels)


        # - pass the fake images through the discriminator
        # - calculate the loss (by passing the output of the discriminator through the criterion with labels set to 1 (real images
        # - update the weights of the generator

        opt_generator.zero_grad()
        loss_generator.backward()
        opt_generator.step()
        epoch_generator_loss += loss_generator.item()
        num_batches += 1


        # print the progress
        print(f"\rEpoch [{epoch}/{numEpochs}] Batch {batch_idx}/{len(loader)}     Loss discriminator: {loss_discriminator:.4f}, loss generator: {loss_generator:.4f}", end="")

        # Log the losses and example images to tensorboard
        if batch_idx % logStep == 0:
            with torch.no_grad():
                # Generate noise via Generator, we always use the same noise to see the progression
                fake = generator(fixed_noise).reshape(-1, 1, 28, 28)
                # Get real data
                data = real.reshape(-1, 1, 28, 28)
                # make grid of pictures and add to tensorboard
                imgGridFake = torchvision.utils.make_grid(fake, normalize=True)
                imgGridReal = torchvision.utils.make_grid(data, normalize=True)

                # TODO: add the images and losses to tensorboard
                # HINT: use the SummaryWriter to add the images and scalars to tensorboard
                # HINT: use the `add_image` method to add the images to tensorboard
                # HINT: use the `add_scalar` method to add the losses to tensorboard
                writer_fake.add_image("Fake Images", imgGridFake, global_step=step)
                writer_real.add_image("Real Images", imgGridReal, global_step=step)

                writer_fake.add_scalar("Generator Loss",loss_generator.item(),global_step=step)
                writer_real.add_scalar("Discriminator Loss",loss_discriminator.item(),global_step=step)
                
                # increment step
                step += 1
    # Early stopping check
    avg_generator_loss = epoch_generator_loss / num_batches

    print(
        f"\nEpoch {epoch}: "
        f"Avg Generator Loss = {avg_generator_loss:.4f}"
    )


    if avg_generator_loss < best_generator_loss:
        best_generator_loss = avg_generator_loss
        patience_counter = 0

        best_generator_state = copy.deepcopy(generator.state_dict())
        best_discriminator_state = copy.deepcopy(discriminator.state_dict())
    else:
        patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break
writer_fake.close()
writer_real.close()

Started Training and visualization...

Epoch [0/100] Batch 1874/1875     Loss discriminator: 0.0697, loss generator: 3.3281
Epoch 0: Avg Generator Loss = 3.5806

Epoch [1/100] Batch 1874/1875     Loss discriminator: 0.1889, loss generator: 2.7440
Epoch 1: Avg Generator Loss = 3.1701

Epoch [2/100] Batch 1874/1875     Loss discriminator: 0.6630, loss generator: 1.8287
Epoch 2: Avg Generator Loss = 3.0429

Epoch [3/100] Batch 1874/1875     Loss discriminator: 0.1128, loss generator: 5.2307
Epoch 3: Avg Generator Loss = 3.9975

Epoch [4/100] Batch 1874/1875     Loss discriminator: 0.3704, loss generator: 3.7694
Epoch 4: Avg Generator Loss = 3.8257

Epoch [5/100] Batch 1874/1875     Loss discriminator: 0.2678, loss generator: 2.4548
Epoch 5: Avg Generator Loss = 3.0753

Epoch [6/100] Batch 1874/1875     Loss discriminator: 0.1268, loss generator: 4.2427
Epoch 6: Avg Generator Loss = 3.5487

Epoch [7/100] Batch 1874/1875     Loss discriminator: 0.4006, loss generator: 2.9392
Epoch 7: Avg Ge

In [58]:
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6007 (pid 347287), started 0:00:28 ago. (Use '!kill 347287' to kill it.)